# PROC FACTMAC による施設・診療科別の患者体験評価モデリング


## エグゼクティブサマリー

複数施設を持つ医療システムは、患者満足度スコアを左右する医療施設と診療科の**交互作用構造**を学習し、過小評価・過大評価となる施設／診療科の組み合わせを特定したいと考えている。この実例は **PROC FACTMAC** で因子分解機を当てはめ、`facility`（施設）と `specialty`（診療科）を2つの名義特徴フィールドとして、1〜10の満足度スコアを区間ターゲットとして扱い、再構成された評価を評価する。

100件のシミュレートされた受診記録において、モデルは**訓練用 R-Square 0.516**（平均絶対誤差 0.95 評価点、RASE 1.25）に達し、予測されたセル平均は最も強い組み合わせと最も弱い組み合わせを明確に分離する ― 最上位が `NorthClinic`／`Cardiology`、最下位が `SouthClinic`／`Cardiology` ― これは約6.8の総平均に収束するのではなく、埋め込まれた交互作用を回復していることを示す。


## データソース

すべてのデータは DATA ステップ（`call streaminit(20240531)` + `rand()`）によりインラインで生成されるため、この実例は外部ファイルやネットワークアクセスを必要とせず完全に自己完結している。この環境はライセンスなしで動作し、出力は100観測に制限されるため、計画は**100件の受診記録内**で因子分解機を実証できるようサイズ決めされている：3施設×2診療科（6セル、各セル平均約17件の受診 ― 確率的勾配降下法が構造を回復するのに十分なシグナル）。

**WORK.ENCOUNTERS** ― 100件の合成患者受診記録（1受診1行）。

| 変数 | 型 | 役割 | 説明 |
|----------|------|------|-------------|
| `facility` | char(12) | 入力（名義） | 医療施設 ― `NorthClinic`、`CentralMed`、`SouthClinic` のいずれか |
| `specialty` | char(14) | 入力（名義） | 臨床診療科 ― `Cardiology` または `Oncology` |
| `satisfaction` | num | ターゲット（区間） | 1〜10段階の患者体験評価。施設バイアス＋診療科バイアス＋真の施設×診療科交互作用項＋ガウスノイズから生成し、[1,10] にクリップして0.1単位に丸めたもの |

潜在的な設計には、施設ごと・診療科ごとの明確に分離されたバイアスに加え、強い交互作用効果が埋め込まれているため、因子分解機は施設のみ、または診療科のみの平均では見逃してしまう構造を回復できるはずである。


# PROC FACTMAC による患者体験評価のモデリング

複数施設を持つ医療システムは、多数の**医療施設**と臨床**診療科**にわたって患者満足度スコアを収集する。施設別または診療科別の平均だけでは、興味深いシグナルが隠れてしまう：ある診療科はある施設では優れているが、別の施設では苦戦するかもしれない。**因子分解機**は、すべての施設とすべての診療科について潜在因子を学習し、各評価を全体平均＋各特徴量の効果＋それらの交互作用としてモデル化することで、まさにこの種のペアワイズ交互作用を捉える。

`PROC FACTMAC` は確率的勾配降下法でこのモデルを当てはめる。この実例では：

1. 100観測環境に合わせてサイズ決めした、施設×診療科の交互作用を埋め込んだ合成受診記録テーブルを生成する。
2. `PROC FACTMAC` で因子分解機を当てはめ、スコア化予測と反復履歴を要求する。
3. 再構成誤差を評価し、モデルが最強・最弱と判定する施設×診療科の組み合わせを明らかにする。


## ステップ1 - 合成患者体験データの生成

3施設×2診療科にわたって100件の受診記録をシミュレートする。各施設と各診療科は隠れたバイアスを持ち、特定の施設／診療科の組み合わせが個々のバイアスから予測されるより高く、または低く評価されるよう、真の**交互作用**項を追加する。ガウスノイズ（標準偏差0.25）が評価をリアルにし、1〜10のスケールにクリップして小数点第1位に丸める。`call streaminit` のシードによりデータは再現可能である。


In [1]:
データ encounters;
    呼出 streaminit(20240531);
    長さ facility $12 specialty $14;

    /* Named facilities and clinical service lines */
    配列 facs[3] $12 _temporary_ (
        'NorthClinic' 'CentralMed' 'SouthClinic');
    配列 specs[2] $14 _temporary_ (
        'Cardiology' 'Oncology');

    /* Hidden per-facility and per-specialty rating biases */
    配列 f_bias[3] _temporary_ (1.5 0.0 -1.5);
    配列 s_bias[2] _temporary_ (1.0 -1.0);

    繰返 enc = 1 から 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Genuine facility x specialty interaction term */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Keep on a 1-10 patient-experience scale */
        もし satisfaction > 10 なら satisfaction = 10;
        もし satisfaction < 1  なら satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        出力;
    終了;

    保持 facility specialty satisfaction;
実行;


NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## ステップ2 - 生の評価分布を確認する

モデリングの前に、合成評価が妥当な挙動をしていることを確認し、施設×診療科セルごとの平均を確認する。`PROC MEANS` のパーセンタイルキーワード（`P25`、`MEDIAN`、`P75`）が全体の散らばりを要約する。2回目の呼び出しは、因子分解機が回復しようとする交互作用を含むセル平均を示す。


In [2]:
処理 平均 データ=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    変数 satisfaction;
    見出 satisfaction='満足度';
実行;

処理 平均 データ=encounters mean NWAY maxdec=2;
    分類 facility specialty;
    変数 satisfaction;
    見出 facility='施設' specialty='診療科' satisfaction='満足度';
実行;


                                                  The MEANS Procedure

 Variable      Label             N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 -------------------------------------------------------------------------------------------------------------------------------
 satisfaction  満足度             100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 -------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                      Analysis Variable : satisfaction 満足度

                                                                     N
                                      施設             診療科           Obs       Mean
                                      -------------------------------------------
                              


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ3 - 因子分解機を当てはめる

`satisfaction`（満足度）を区間**ターゲット**として、`facility`（施設）と `specialty`（診療科）の2つのカテゴリカルフィールドを名義**入力**特徴量としてモデル化する。主なオプション：

- `LEARN=0.02` - 確率的勾配降下法のステップ幅。この小規模で標準化された計画では、控えめな学習率がセル構造を当てはめつつ最適化を安定させる。
- `L2=0.0005` - 軽い L2 正則化で、因子が総平均へ過度に縮小しないようにするのに十分な強さ。
- `SEED=20240531` - 再現可能な初期化。
- `OUT=scored` - 行ごとの予測（`P_satisfaction`）を書き出す。
- `OUTSTAT=fitstats` - 反復ごとの RMSE 履歴を取得し、収束を確認できるようにする。

`ID` ステートメントは主要フィールドをスコア化出力にコピーし、各予測がその施設と診療科でラベル付けされたままになるようにする。


In [3]:
処理 factmac データ=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    入力 facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    見出 facility='施設' specialty='診療科' satisfaction='満足度';
実行;



                         The FACTMAC Procedure

  Target variable: 満足度
  Input variable: 施設
  Input variable: 診療科

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## ステップ4 - 収束を確認する

`OUTSTAT=` テーブルは各 SGD 反復での訓練 RMSE を記録する。単調に減少し横ばいになる曲線は、モデルが（デフォルト100の）反復予算内で収束したことを示す。


In [4]:
処理 印刷 データ=fitstats(obs=15) 見出 noobs;
実行;


ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## ステップ5 - 再構成誤差を評価する

スコア化テーブルは、観測された `satisfaction`（満足度）とモデルの `P_satisfaction` を並べて保持する。残差と絶対誤差を導出し、要約する。残差がほぼゼロを中心とし、予測評価の散らばりが（総平均でのフラットな線に収束するのではなく）観測された散らばりに近づいていることは、因子分解機が埋め込まれた施設×診療科構造を再現していることを示す。


In [5]:
データ resid;
    設定 scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
実行;

処理 印刷 データ=scored(obs=10) 見出 noobs;
実行;

処理 平均 データ=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    変数 satisfaction p_satisfaction residual abs_err;
    見出 satisfaction='満足度' p_satisfaction='予測満足度' residual='残差' abs_err='絶対誤差';
実行;


   FACILITY   SPECIALTY  SATISFACTION  P_SATISFACTION
-----------  ----------  ------------  --------------
SouthClinic  Oncology             6.3    6.1577265381
NorthClinic  Oncology             5.4    6.0430846516
CentralMed   Cardiology           7.9    9.5419769923
SouthClinic  Cardiology           4.7    5.8019161993
CentralMed   Oncology             6.2    5.9284427651
NorthClinic  Cardiology            10    7.6719465958
NorthClinic  Oncology             5.9    6.0430846516
NorthClinic  Cardiology            10    7.6719465958
SouthClinic  Cardiology           4.9    5.8019161993
CentralMed   Oncology             6.2    5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                   N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ------------------------------------------------------------------------------------------------


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ6 - 施設×診療科別のパフォーマンスを可視化する

品質改善チームにとって実用的な視点は、**施設×診療科の組み合わせ**別の予測評価である。モデル予測の体験がシステム全体の平均を大きく下回る組み合わせはレビュー候補となる。絶対誤差列は、モデルがきれいに当てはまる箇所と、線形因子分解機が到達できる上限をクリップされた1〜10の上限が制限している箇所を示す。


In [6]:
処理 平均 データ=resid mean NWAY maxdec=3;
    分類 facility specialty;
    変数 p_satisfaction abs_err;
    見出 facility='施設' specialty='診療科' p_satisfaction='予測満足度' abs_err='絶対誤差';
実行;


                                                  The MEANS Procedure

                                      Analysis Variable : p_satisfaction 予測満足度

                                                                     N
                                      施設             診療科           Obs       Mean
                                      -------------------------------------------
                                      CentralMed     Cardiology     13      9.542

                                                     Oncology       20      5.928

                                      NorthClinic    Cardiology     18      7.672

                                                     Oncology       16      6.043

                                      SouthClinic    Cardiology     20      5.802

                                                     Oncology       13      6.158
                                      -------------------------------------------

                                  


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 結果の解釈

`OUTSTAT=` からの反復履歴は、訓練 RMSE が初回パスの約1.68からおよそ7回目の反復までに**1.265**近辺の横ばいへと低下することを示しており、SGD の当てはめが反復予算内で十分に収束したことを裏付ける。Fit Statistics は**訓練 R-Square 0.516**、**平均絶対誤差 0.954** 評価点、**RASE 1.25** を報告する ― 因子分解機は標準偏差1.81を持つ1〜10の満足度スコアの分散のおよそ半分を説明しており、総平均を予測しているのではなく、真に構造を学習していることになる。残差の要約もこれを裏付ける：残差平均はほぼゼロ（0.020）であり、予測評価は5.80〜9.54（標準偏差1.27）の範囲に及び、観測された散らばりに（完全ではないが）近い動きをしている。

施設×診療科テーブルは、潜在モデルを医療品質チームが行動に移せるものに変える。モデルは `CentralMed`／`Cardiology` を最高位（予測9.54）、`SouthClinic`／`Cardiology` を最低位（予測5.80）と評価し、Cardiology があるサイトでは優れているが別のサイトでは弱いという埋め込まれた交互作用を回復している。絶対誤差列はモデルの限界について正直である：2つの Oncology セルはほぼ正確に再現される（平均絶対誤差 0.19〜0.24）一方、`NorthClinic`／`Cardiology` は過小予測されている（誤差2.33）。これは、その真の評価がクリップされた1〜10の上限に集中しており、線形因子分解機がそこに到達できないためである。

**次のステップ**として実務者が取り得るのは、過学習を防ぐための `PARTITION` ホールドアウトの追加、バイアスと分散のトレードオフを調整するための `LEARN=` と `L2=` のチューニング、または（提供者、受診種別、季節などの）特徴量セットの拡張により、因子分解機がより高次の体験ドライバーをモデル化できるようにすることである。完全ライセンス環境では、セルあたりより多くの受診記録を持つより大きな施設×診療科グリッドにより、モデルはここで示した6セル計画よりも細かい交互作用構造を解決できるだろう。
